# Data quality and validation with Great Expectations
**Question: Can we trust this housing data before using it?**

A CSV can load successfully and still contain **missing areas, negative prices, or incorrect columns**. Today we will turn our assumptions into repeatable checks.

By the end you can:
- Explain what Great Expectations checks and why it helps.
- Check schema, data types, missing values, and simple business rules.
- Read a validation result.
- Stop a pipeline before invalid data reaches feature engineering.

Use the existing `.venv` Python kernel and run cells in order. No cloud account is needed.

In [1]:
# Locate our files whether the notebook starts in the project root or data_valid.

from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name != "data_valid":
    ROOT = ROOT / "data_valid"
assert (ROOT / "get_data.py").exists(), "Start in the project root or data_valid folder."
sys.path.insert(0, str(ROOT))

import great_expectations as gx

print("ROOT:", ROOT)
print("Python:", sys.executable)
print("Great Expectations:", gx.__version__)

ROOT: C:\Users\Ash\Desktop\educational\play_ML\data_valid
Python: C:\Users\Ash\Desktop\educational\play_ML\.venv\Scripts\python.exe
Great Expectations: 1.24.1


## 1. Generate and inspect raw housing data
Each row describes one house. We have five columns: an ID, area, bedrooms, year built, and a price target. The generator also makes copies with deliberate mistakes.

This tutorial is about checking the data, so we do not train a model. The same checks could run before training or before data enters a feature store.

In [5]:
from get_data import get_data

get_data()

# Let pandas read each file normally; do not silently coerce or repair bad data.
clean = pd.read_csv(ROOT / "data/houses_clean.csv")
dirty = pd.read_csv(ROOT / "data/houses_dirty.csv")

display("clean:", clean.head())
display("clean data type:", clean.dtypes.rename("pandas dtype").to_frame())

Created four small CSV examples in data/ (100 rows each).


'clean:'

,house_id,area_sqft,bedrooms,year_built,sale_price
0,1,905.0,5,1994,245427.84
1,2,2480.0,1,2016,495656.39
2,3,2205.0,5,1999,474630.92
3,4,1709.0,1,2007,338334.77
4,5,1695.0,4,1998,357702.54


'clean data type:'

,pandas dtype
house_id,int64
area_sqft,float64
bedrooms,int64
year_built,int64
sale_price,float64


## 2. What does data quality mean here?

- **Schema** describes the table's structure: column names and data types.
- **Missing-value checks** ask whether required cells contain values.
- **Business rules** ask whether the values are sensible for this application.

| Assumption | Check |
| --- | --- |
| We receive all five expected columns | Schema check |
| Price is numeric | Type check |
| Every house has an area | Missing-value check |
| Bedrooms is between 1 and 10 | Range check |
| Each house appears once | Unique ID check |

The allowed ranges are classroom assumptions. A production team should agree on rules with the data owners.

First, inspect the mistakes manually. This is useful for exploration, but it becomes repetitive for every new file.

In [6]:
display(dirty.head(7))

# Two missing areas are visible here. Other problems require other rules.
display(dirty.isna().sum().rename("missing values").to_frame())

,house_id,area_sqft,bedrooms,year_built,sale_price
0,1,NaN,5,1994,245427.84
1,2,NaN,1,2016,495656.39
2,3,2205.0,5,1999,-50000.00
3,4,1709.0,0,2007,338334.77
4,6,1695.0,4,1998,357702.54
5,6,2674.0,4,1990,535429.54
6,7,897.0,4,2010,239029.74


,missing values
house_id,0
area_sqft,2
bedrooms,0
year_built,0
sale_price,0


## 3. What is an Expectation?
Great Expectations (GX) is a library for testing data against explicit rules.
An **Expectation** is one rule. Here we require every house to have an area.

The setup below creates a temporary GX workspace and connects it to our pandas data. A **Batch** is simply the set of rows we want to check; here it is the whole dataframe.

In [8]:
# Context = GX workspace. "ephemeral" keeps setup in memory.
context = gx.get_context(mode="ephemeral")
context.enable_analytics(False)
context.variables.progress_bars = {"globally": False}

# Source -> Asset -> Batch Definition tells GX how to receive our dataframe.
source = context.data_sources.add_pandas(name="lesson_source")
asset = source.add_dataframe_asset(name="house_prices")
batch_definition = asset.add_batch_definition_whole_dataframe(name="all_rows")

# Define one reusable rule, not a check tied to a single CSV file.
area_required = gx.expectations.ExpectColumnValuesToNotBeNull(column="area_sqft")

clean_batch = batch_definition.get_batch(batch_parameters={"dataframe": clean})
clean_check = clean_batch.validate(area_required)

print("Clean file passed the area rule:", clean_check.success)

Clean file passed the area rule: True


In [9]:
# Reuse exactly the same rule on a different delivery.

dirty_batch = batch_definition.get_batch(batch_parameters={"dataframe": dirty})
dirty_check = dirty_batch.validate(area_required)
print("Dirty file passed the area rule:", dirty_check.success)
print("Rows with missing area:", dirty_check.result["unexpected_count"])

# The result reports the problem. The dataframe itself has not been changed.
assert not dirty_check.success
assert dirty.area_sqft.isna().sum() == 2

Dirty file passed the area rule: False
Rows with missing area: 2


## 4. One check is not enough
A missing-value check will not detect a negative price. A type check will not tell us whether zero bedrooms is acceptable.

We also need to know a subtle detail: numeric range checks usually ignore missing values. A valid range result does not replace an explicit missing-value check.

In [11]:
# The non-missing areas are within range, even though two areas are absent.
area_range = gx.expectations.ExpectColumnValuesToBeBetween(
    column="area_sqft", min_value=300, max_value=10000,
)
range_check = dirty_batch.validate(area_range)

print("Area range check:",         range_check.success)
print("Area missing-value check:", dirty_check.success)
print("Both rules are needed.")

Area range check: True
Area missing-value check: False
Both rules are needed.


## 5. Reuse a suite of rules
An **Expectation Suite** is a collection of rules. Open `expectations.py` and read `make_suite()`.

It checks:
- Exactly five expected columns, in any order, and at least one row.
- No missing values in any required column.
- Integer IDs, bedrooms, and years; decimal areas and prices.
- Reasonable ranges for area, bedrooms, year built, and price.
- Unique house IDs.

`validate_data()` connects our dataframe to this suite and returns the GX result. A **Validation Definition** is the object linking the data batch definition and the suite. We put that setup in a reusable function so the notebook and pipeline use the same rules.

In [12]:
from expectations import make_suite
from validate_data import validate_data, result_table, save_report

suite = make_suite()
print("Rules in our suite:", len(suite.expectations))

clean_result = validate_data(clean)
print("All clean-data rules passed:", clean_result.success)
display(result_table(clean_result))

Rules in our suite: 17
All clean-data rules passed: True


,rule,column,passed,unexpected_count,note
0,expect_table_columns_to_match_set,(whole table),True,NaN,
1,expect_table_row_count_to_be_between,(whole table),True,NaN,
2,expect_column_values_to_not_be_null,house_id,True,0.0,
3,expect_column_values_to_be_of_type,house_id,True,NaN,
4,expect_column_values_to_be_unique,house_id,True,0.0,
5,expect_column_values_to_not_be_null,area_sqft,True,0.0,
6,expect_column_values_to_be_of_type,area_sqft,True,NaN,
7,expect_column_values_to_be_between,area_sqft,True,0.0,
8,expect_column_values_to_not_be_null,bedrooms,True,0.0,
9,expect_column_values_to_be_of_type,bedrooms,True,NaN,


In [13]:
dirty_result = validate_data(dirty)
print("All dirty-data rules passed:", dirty_result.success)

# Show only failed checks to make the report easy to discuss.
failed_checks = result_table(dirty_result).query("passed == False")
display(failed_checks)

report_path = save_report(dirty_result, "notebook_dirty")
print("Detailed JSON report:", report_path)

All dirty-data rules passed: False


,rule,column,passed,unexpected_count,note
4,expect_column_values_to_be_unique,house_id,False,2.0,
5,expect_column_values_to_not_be_null,area_sqft,False,2.0,
10,expect_column_values_to_be_between,bedrooms,False,1.0,
16,expect_column_values_to_be_between,sale_price,False,1.0,


Detailed JSON report: C:\Users\Ash\Desktop\educational\play_ML\data_valid\reports\notebook_dirty.json


Expected failures: two missing areas, a negative price, zero bedrooms, and a duplicated house ID. The uniqueness rule counts both occurrences of the duplicate.

A blank/NaN `unexpected_count` means GX did not provide that metric for the rule. Always read `passed` to decide whether a rule succeeded. A missing column can also make column-level checks impossible to evaluate; those results remain failures, not passes.

## 6. Schema validation: missing columns and wrong types
A producer might remove or rename a column. That changes the schema even if all remaining cells are filled.

A price column can also contain text. Loading the CSV may then make the whole column an `object` rather than a numeric dtype. Our numeric type expectation will fail. Type expectations in this example match pandas' inference for these CSV files.

In [14]:
bad_schema = pd.read_csv(ROOT / "data/houses_bad_schema.csv")
print("Received columns:", bad_schema.columns.tolist())

schema_result = validate_data(bad_schema)
display(result_table(schema_result).query("passed == False"))

assert not schema_result.success

Received columns: ['house_id', 'area_sqft', 'year_built', 'sale_price']


,rule,column,passed,unexpected_count,note
0,expect_column_values_to_not_be_null,bedrooms,False,NaN,
1,expect_column_values_to_be_between,bedrooms,False,NaN,
2,expect_table_columns_to_match_set,(whole table),False,NaN,
10,expect_column_values_to_be_of_type,bedrooms,False,NaN,Could not evaluate; inspect JSON details


In [15]:
bad_type = pd.read_csv(ROOT / "data/houses_bad_type.csv")
display(bad_type[["sale_price"]].head())
print("Price dtype:", bad_type.sale_price.dtype)

type_result = validate_data(bad_type)
display(result_table(type_result).query("passed == False"))

assert not type_result.success

,sale_price
0,unknown
1,495656.39
2,474630.92
3,338334.77
4,357702.54


Price dtype: object


,rule,column,passed,unexpected_count,note
0,expect_column_values_to_be_between,sale_price,False,NaN,
16,expect_column_values_to_be_of_type,sale_price,False,100.0,


## 7. Put validation into a pipeline
```text
Raw CSV -> validate -> PASS -> prepare a feature -> save for later training
                   -> FAIL -> report the problem and stop
```

`transform_data.py` adds `area_per_bedroom = area_sqft / bedrooms`. We validate first to avoid missing areas or division by zero.

The important part of `run_pipeline.py` is:
```python
result = validate_data(data)
if not result.success:
    return False
prepared = transform_data(data)
```
GX supplies a result. **Our pipeline decides to stop.** A failed rule does not automatically stop arbitrary Python code unless we act on it.

In [11]:
from run_pipeline import run_pipeline

# A clean delivery reaches feature engineering and produces an output file.
assert run_pipeline(ROOT / "data/houses_clean.csv")
prepared = pd.read_csv(ROOT / "output/houses_clean_features.csv")

display(prepared.head())

                                rule        column  passed  unexpected_count note
   expect_table_columns_to_match_set (whole table)    True               NaN     
expect_table_row_count_to_be_between (whole table)    True               NaN     
 expect_column_values_to_not_be_null      house_id    True               0.0     
  expect_column_values_to_be_of_type      house_id    True               NaN     
   expect_column_values_to_be_unique      house_id    True               0.0     
 expect_column_values_to_not_be_null     area_sqft    True               0.0     
  expect_column_values_to_be_of_type     area_sqft    True               NaN     
  expect_column_values_to_be_between     area_sqft    True               0.0     
 expect_column_values_to_not_be_null      bedrooms    True               0.0     
  expect_column_values_to_be_of_type      bedrooms    True               NaN     
  expect_column_values_to_be_between      bedrooms    True               0.0     
 expect_column_v


Report: C:\Users\Ash\Desktop\educational\play_ML\data_valid\reports\houses_clean.json
PASS: prepared data for later model training: C:\Users\Ash\Desktop\educational\play_ML\data_valid\output\houses_clean_features.csv


,house_id,area_sqft,bedrooms,year_built,sale_price,area_per_bedroom
0,1,905.0,5,1994,245427.84,181.00
1,2,2480.0,1,2016,495656.39,2480.00
2,3,2205.0,5,1999,474630.92,441.00
3,4,1709.0,1,2007,338334.77,1709.00
4,5,1695.0,4,1998,357702.54,423.75


In [12]:
# False and a STOP message are EXPECTED here; the notebook should continue.
# This delivery does not reach feature engineering or create a new training file.
assert not run_pipeline(ROOT / "data/houses_dirty.csv")

                                rule        column  passed  unexpected_count note
   expect_table_columns_to_match_set (whole table)    True               NaN     
expect_table_row_count_to_be_between (whole table)    True               NaN     
 expect_column_values_to_not_be_null      house_id    True               0.0     
  expect_column_values_to_be_of_type      house_id    True               NaN     
   expect_column_values_to_be_unique      house_id   False               2.0     
 expect_column_values_to_not_be_null     area_sqft   False               2.0     
  expect_column_values_to_be_of_type     area_sqft    True               NaN     
  expect_column_values_to_be_between     area_sqft    True               0.0     
 expect_column_values_to_not_be_null      bedrooms    True               0.0     
  expect_column_values_to_be_of_type      bedrooms    True               NaN     
  expect_column_values_to_be_between      bedrooms   False               1.0     
 expect_column_v

## 8. What should happen after a failure?
Inspect the validation result, identify the source problem, and correct or request the data using an agreed policy. Then validate again. Do not silently replace missing prices or remove rows just to obtain a pass.

GX does not automatically clean data. Passing the suite means the table satisfies these rules; it does not prove every price is true or guarantee an accurate model.

Our setup is intentionally small: in-memory GX configuration, local CSV files, reusable Python rules, and saved JSON reports. No cloud service or model deployment is needed.

## Try it yourself
1. Copy `clean` and set one `sale_price` to zero. Which expectation should fail?
2. Remove the `year_built` column. Is this a schema problem or a missing cell?
3. Set one bedroom count to 20. Is its type still correct? Is its value acceptable?

Discuss why a rule can be useful even when another rule passes.

In [13]:
# One exercise solution: change a value without changing the original data.
exercise = clean.copy()
exercise.loc[0, "sale_price"] = 0
exercise_result = validate_data(exercise)
display(result_table(exercise_result).query("passed == False"))
assert not exercise_result.success

,rule,column,passed,unexpected_count,note
16,expect_column_values_to_be_between,sale_price,False,1.0,


You have used Great Expectations to define rules, detect bad deliveries, read results, and protect the next pipeline step.

See `README.md` for terminal commands. To run validation alone:
```powershell
python validate_data.py data/houses_dirty.csv
```
A failing input returns exit code 1 on purpose, so an automated job can detect it.

Official references: [GX introduction](https://docs.greatexpectations.io/docs/core/introduction/try_gx/) and [Expectations](https://docs.greatexpectations.io/docs/core/define_expectations/).